In [ ]:
# Load API keys from .env (kept out of git)
import os, pathlib
for _d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    _e = _d / '.env'
    if _e.exists():
        for _l in _e.read_text().splitlines():
            if _l.strip() and not _l.startswith('#') and '=' in _l:
                _k,_v=_l.split('=',1); os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))
        break


This code uses news articles to find promising stocks in the selected industry and then uses stock data to find how much to invest in each stock. Finally, the last code provides percent return for each stock.

In [ ]:
# Enhanced Investment Analysis Pipeline using NY Times News Data
from openai import OpenAI
import json
import pandas as pd
import yfinance as yf
from datetime import datetime, timedelta
import os
import requests
import time
import re

def enhanced_investment_analysis_pipeline(api_key_openai, user_input_keyword, investment_amount, reinvestment_amount=0,
                                        news_data_path=None, alpha_vantage_key=None):
    """
    Enhanced investment analysis pipeline that uses NY Times news data and returns comprehensive company analysis
    """

    # Initialize OpenAI client
    client = OpenAI(api_key=api_key_openai)

    # 1. Load NY Times News Data from CSV
    def load_news_data_from_csv(file_path=None):
        """Load news data from the NY Times CSV files - with Google Drive support"""

        # Check if we're in Google Colab and mount Drive if needed
        try:
            from google.colab import drive
            COLAB_ENVIRONMENT = True

            # Mount Google Drive
            try:
                drive.mount('/content/drive')
                print("✅ Google Drive mounted successfully!")
            except Exception as e:
                print(f"⚠️ Drive already mounted or mount failed: {e}")

        except ImportError:
            COLAB_ENVIRONMENT = False
            print("ℹ Not in Google Colab environment")

        if file_path is None:
            # Try to find the news data files in common locations
            if COLAB_ENVIRONMENT:
                # Colab + Google Drive paths
                possible_paths = [
                    "/content/drive/MyDrive/Investment_News_Data/investment_news_data.csv",
                    "/content/drive/My Drive/Investment_News_Data/investment_news_data.csv",
                    "/content/drive/MyDrive/investment_news_data.csv",
                    "/content/drive/My Drive/investment_news_data.csv",
                    "investment_news_data.csv",  # Local fallback
                    "./investment_news_data.csv"
                ]
            else:
                # Local environment paths
                possible_paths = [
                    "investment_news_data.csv",
                    "./investment_news_data.csv",
                    "../investment_news_data.csv",
                    "c:/Users/jathi/Downloads/investment_news_data.csv"  # Download folder
                ]

            print(f"🔍 Searching for news data in {len(possible_paths)} possible locations...")
            for i, path in enumerate(possible_paths, 1):
                print(f"  {i}. Checking: {path}")
                if os.path.exists(path):
                    file_path = path
                    print(f"  ✅ Found at: {path}")
                    break
                else:
                    print(f"  ❌ Not found")

        if file_path is None or not os.path.exists(file_path):
            print("\n❌ News data CSV file not found!")
            print("📋 Please ensure:")
            if COLAB_ENVIRONMENT:
                print("1. You've run the Untitled0.ipynb notebook to collect NY Times data")
                print("2. The data was saved to Google Drive")
                print("3. Check your Google Drive folder: /MyDrive/Investment_News_Data/")
                print("4. The file 'investment_news_data.csv' exists in that folder")
            else:
                print("1. You've run the Untitled0.ipynb notebook")
                print("2. The CSV files are in the same directory as this notebook")
                print("3. Or provide the full path to the CSV file")
            return None

        try:
            df = pd.read_csv(file_path)
            print(f"✅ Successfully loaded {len(df)} articles from:")
            print(f"   📁 {file_path}")

            # Show data info
            if len(df) > 0:
                industries = df['industry'].value_counts()
                print(f"📊 Data breakdown:")
                for industry, count in industries.items():
                    print(f"   - {industry.title()}: {count} articles")

            return df
        except Exception as e:
            print(f"❌ Error loading news data from {file_path}: {e}")
            return None

    # Load the news data
    news_df = load_news_data_from_csv(news_data_path)
    if news_df is None:
        return None

    # 2. Process News Data for Analysis
    def process_news_data(df, keyword):
        """Process and filter news data based on the keyword"""
        # Filter by industry if keyword matches
        industry_mapping = {
            'technology': 'technology',
            'tech': 'technology',
            'energy': 'energy',
            'agriculture': 'agriculture',
            'farming': 'agriculture'
        }

        target_industry = industry_mapping.get(keyword.lower())

        if target_industry:
            filtered_df = df[df['industry'] == target_industry]
            print(f"🎯 Filtered to {len(filtered_df)} {target_industry} articles")
        else:
            # Use all articles if no specific industry match
            filtered_df = df
            print(f"📰 Using all {len(filtered_df)} articles")

        # Create comprehensive article string for analysis
        articles_string = ""
        for _, article in filtered_df.iterrows():
            articles_string += f"Date: {article['pub_date']}\n"
            articles_string += f"Industry: {article['industry']}\n"
            articles_string += f"Headline: {article['headline']}\n"
            articles_string += f"Content: {article['snippet']}\n"
            articles_string += f"URL: {article['web_url']}\n"
            articles_string += "---\n"

        return articles_string, filtered_df

    # Process the news data
    all_articles_string, filtered_news = process_news_data(news_df, user_input_keyword)

    # 3. Enhanced OpenAI Analysis for Company Identification
    enhanced_system_prompt = """
    You are an expert financial analyst specializing in identifying high-potential public companies for long-term investment.

    Using ONLY the provided news articles, analyze market trends, technological developments, regulatory changes,
    and company-specific information to identify the most promising publicly-traded companies for a 10-year investment horizon.

    Focus on:
    - Companies with strong competitive moats and market positions
    - Emerging growth trends and technological disruptions
    - Regulatory tailwinds and policy support
    - Strong financial fundamentals and growth potential
    - ESG considerations and sustainability factors

    IMPORTANT: Only recommend publicly-traded companies with valid stock tickers on major exchanges (NYSE, NASDAQ).
    Exclude private companies, startups without public listings, and any non-investable entities.

    Return your analysis as a properly formatted JSON object with the following exact structure:
    """

    enhanced_user_prompt = f"""
    Keyword Focus: {user_input_keyword}

    News Articles Data:
    {all_articles_string}

    Based on this news data, provide a comprehensive investment analysis with 15-25 high-potential companies.

    Return ONLY a valid JSON object with this exact structure:
    {{
        "analysis_summary": {{
            "market_themes": ["theme1", "theme2", "theme3"],
            "key_trends": ["trend1", "trend2", "trend3"],
            "implementation_date": "2024-01-01",
            "total_companies_analyzed": 20
        }},
        "recommended_companies": [
            {{
                "ticker": "AAPL",
                "company_name": "Apple Inc.",
                "sector": "Technology",
                "investment_thesis": "Detailed explanation of why this company was selected based on news analysis",
                "growth_catalysts": ["catalyst1", "catalyst2"],
                "risk_factors": ["risk1", "risk2"],
                "confidence_score": 8.5,
                "recommended_allocation": 12.5,
                "price_target_10yr": "Conservative estimate based on analysis"
            }}
        ],
        "portfolio_strategy": {{
            "diversification_approach": "Strategy description",
            "sector_allocation": {{
                "technology": 40,
                "healthcare": 25,
                "energy": 20,
                "other": 15
            }},
            "risk_level": "Moderate-Aggressive",
            "expected_annual_return": "8-12%"
        }}
    }}
    """

    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": enhanced_system_prompt},
                {"role": "user", "content": enhanced_user_prompt}
            ],
            temperature=0.3  # Lower temperature for more consistent JSON output
        )

        response_content = completion.choices[0].message.content
        print("✅ Received analysis from OpenAI")

        # Clean and parse JSON response
        # Remove any markdown formatting if present
        if "```json" in response_content:
            response_content = response_content.split("```json")[1].split("```")[0]
        elif "```" in response_content:
            response_content = response_content.split("```")[1]

        # Parse JSON
        try:
            analysis_result = json.loads(response_content.strip())
            print(f"✅ Successfully parsed JSON with {len(analysis_result.get('recommended_companies', []))} companies")

        except json.JSONDecodeError as e:
            print(f"❌ JSON parsing failed: {e}")
            print("Raw response:", response_content[:500] + "...")
            return None

    except Exception as e:
        print(f"❌ OpenAI API error: {e}")
        return None

    # 4. Validate and Extract Company List
    recommended_companies = analysis_result.get('recommended_companies', [])
    if not recommended_companies:
        print("❌ No companies found in the analysis")
        return None

    # Extract ticker symbols for further analysis
    stock_list = [company['ticker'] for company in recommended_companies if company.get('ticker')]
    print(f"📊 Extracted {len(stock_list)} company tickers: {stock_list[:10]}...")

    # 5. Fetch Stock Data for Validation
    print("📈 Fetching stock data for validation...")
    valid_tickers = []
    stock_data = {}

    end_date = pd.to_datetime('today')
    start_date = end_date - pd.DateOffset(years=5)  # 5 years of data for analysis

    for ticker in stock_list:
        try:
            data = yf.download(ticker, start=start_date, end=end_date, progress=False)
            if not data.empty:
                stock_data[ticker] = data
                valid_tickers.append(ticker)
            else:
                print(f"⚠️ No data found for {ticker}")
        except Exception as e:
            print(f"⚠️ Could not download data for {ticker}: {e}")

    print(f"✅ Validated {len(valid_tickers)} out of {len(stock_list)} tickers")

    # 6. Filter recommended companies to only include valid tickers
    validated_companies = [
        company for company in recommended_companies
        if company.get('ticker') in valid_tickers
    ]

    # Update the analysis result
    analysis_result['recommended_companies'] = validated_companies
    analysis_result['analysis_summary']['validated_companies'] = len(validated_companies)

    # 7. Enhanced Return Analysis (using the finance report methodology)
    def calculate_enhanced_returns(companies, start_date_str="2023-01-01", end_date_str=None):
        """Calculate returns using the validated company list"""
        if end_date_str is None:
            end_date_str = datetime.now().strftime('%Y-%m-%d')

        total_return = 0
        successful_calculations = 0

        for company in companies:
            ticker = company.get('ticker')
            allocation = company.get('recommended_allocation', 0)

            if ticker and allocation > 0:
                try:
                    # Calculate return percentage for this stock
                    stock = yf.Ticker(ticker)
                    hist_data = stock.history(start=start_date_str, end=end_date_str)

                    if not hist_data.empty:
                        start_price = hist_data.iloc[0]['Close']
                        end_price = hist_data.iloc[-1]['Close']
                        stock_return = ((end_price - start_price) / start_price) * 100

                        # Weight by allocation percentage
                        weighted_return = stock_return * (allocation / 100)
                        total_return += weighted_return
                        successful_calculations += 1

                except Exception as e:
                    print(f"⚠️ Return calculation failed for {ticker}: {e}")

        if successful_calculations > 0:
            portfolio_return = total_return
            print(f"📊 Portfolio Return Calculation: {portfolio_return:.2f}%")
            print(f"📊 Based on {successful_calculations} successful calculations")
            return portfolio_return
        else:
            print("❌ No successful return calculations")
            return 0

    # Calculate returns for the portfolio
    portfolio_return = calculate_enhanced_returns(validated_companies)

    # 8. Prepare Final Results
    final_results = {
        'analysis_data': analysis_result,
        'stock_data': stock_data,
        'portfolio_return': portfolio_return,
        'news_articles_analyzed': len(filtered_news),
        'implementation_date': analysis_result.get('analysis_summary', {}).get('implementation_date', '2024-01-01'),
        'validated_tickers': valid_tickers
    }

    # Print summary
    print("\n" + "="*60)
    print("📋 ENHANCED INVESTMENT ANALYSIS SUMMARY")
    print("="*60)
    print(f"🎯 Target Industry: {user_input_keyword}")
    print(f"📰 News Articles Analyzed: {len(filtered_news)}")
    print(f"🏢 Companies Recommended: {len(validated_companies)}")
    print(f"📊 Portfolio Return Estimate: {portfolio_return:.2f}%")
    print(f"💰 Investment Amount: ${investment_amount:,.2f}")

    if reinvestment_amount:
        final_value = investment_amount + (investment_amount * portfolio_return / 100) + reinvestment_amount
        print(f"💎 Projected Value (with reinvestment): ${final_value:,.2f}")
        return final_value
    else:
        final_value = investment_amount + (investment_amount * portfolio_return / 100)
        print(f"💎 Projected Value: ${final_value:,.2f}")
        return final_results

print("✅ Enhanced investment analysis pipeline ready!")
print("🚀 Now uses NY Times news data and returns comprehensive JSON analysis with 15-25 validated companies!")

✅ Enhanced investment analysis pipeline ready!
🚀 Now uses NY Times news data and returns comprehensive JSON analysis with 15-25 validated companies!


In [ ]:
# Enhanced Execution with NY Times Data Integration (Google Drive Support)
import json

# Configuration
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

# Google Drive path for the NY Times news data (set to None for auto-detection)
NEWS_DATA_PATH = None  # Will auto-detect from Google Drive or local storage

# Investment parameters
keyword = "technology"  # Can be: technology, energy, agriculture
initial_investment = 10000
reinvestment_periods = {"quarterly": 2500, "yearly": 10000, "none": 0}

print("🚀 Starting Enhanced Investment Analysis with NY Times Data")
print("🔗 Integrating with Google Drive for data access")
print("="*70)

# Check environment and provide guidance
try:
    from google.colab import drive
    print("📱 Google Colab environment detected")
    print("💾 Will attempt to access NY Times data from Google Drive")
except ImportError:
    print("💻 Local environment detected")
    print("📁 Will search for CSV files in local directories")

print(f"\n🎯 Analyzing {keyword} sector using NY Times news data...")

# Run initial analysis to get company recommendations
initial_results = enhanced_investment_analysis_pipeline(
    api_key_openai=OPENAI_API_KEY,
    user_input_keyword=keyword,
    investment_amount=initial_investment,
    reinvestment_amount=0,
    news_data_path=NEWS_DATA_PATH  # Auto-detect from Drive
)

if initial_results and isinstance(initial_results, dict):
    # Display the comprehensive analysis
    analysis_data = initial_results.get('analysis_data', {})

    print("\n" + "="*70)
    print("📊 COMPREHENSIVE COMPANY ANALYSIS RESULTS")
    print("="*70)

    # Display market themes and trends
    summary = analysis_data.get('analysis_summary', {})
    print(f"🎯 Market Themes: {', '.join(summary.get('market_themes', []))}")
    print(f"📈 Key Trends: {', '.join(summary.get('key_trends', []))}")

    # Display recommended companies
    companies = analysis_data.get('recommended_companies', [])
    print(f"\n🏢 TOP RECOMMENDED COMPANIES ({len(companies)} total):")
    print("-" * 70)

    for i, company in enumerate(companies[:10], 1):  # Show top 10
        print(f"{i:2d}. {company.get('ticker', 'N/A')} - {company.get('company_name', 'N/A')}")
        print(f"    📊 Allocation: {company.get('recommended_allocation', 0)}%")
        print(f"    🎯 Confidence: {company.get('confidence_score', 0)}/10")
        print(f"    💡 Thesis: {company.get('investment_thesis', 'N/A')[:100]}...")
        print()

    # Display portfolio strategy
    portfolio = analysis_data.get('portfolio_strategy', {})
    print("📋 PORTFOLIO STRATEGY:")
    print(f"🎯 Approach: {portfolio.get('diversification_approach', 'N/A')}")
    print(f"⚖️ Risk Level: {portfolio.get('risk_level', 'N/A')}")
    print(f"📈 Expected Return: {portfolio.get('expected_annual_return', 'N/A')}")

    # Display sector allocation
    sector_alloc = portfolio.get('sector_allocation', {})
    if sector_alloc:
        print("\n📊 SECTOR ALLOCATION:")
        for sector, percentage in sector_alloc.items():
            print(f"  {sector.title()}: {percentage}%")

    # Save detailed results to JSON file
    output_filename = f"{keyword}_investment_analysis_results.json"
    try:
        with open(output_filename, 'w', encoding='utf-8') as f:
            json.dump(initial_results, f, indent=2, default=str)
        print(f"\n💾 Detailed results saved to: {output_filename}")

        # If in Colab, also try to save to Drive
        try:
            from google.colab import drive
            drive_path = f"/content/drive/MyDrive/Investment_News_Data/{output_filename}"
            with open(drive_path, 'w', encoding='utf-8') as f:
                json.dump(initial_results, f, indent=2, default=str)
            print(f"💾 Also saved to Google Drive: {drive_path}")
        except:
            pass

    except Exception as e:
        print(f"⚠️ Could not save results file: {e}")

    print("\n" + "="*70)
    print("🎯 NEXT STEPS:")
    print("1. 📊 Review the recommended companies and their analysis")
    print("2. 🏢 Use these tickers for financial data collection (next cell)")
    print("3. 📈 Generate final investment strategy based on financial fundamentals")
    print("4. 💰 Implement portfolio with recommended allocations")

    # Optional: Run reinvestment analysis (simplified to avoid too many API calls)
    print(f"\n🔄 Running sample reinvestment scenario...")
    sample_period = "yearly"
    sample_reinvestment = reinvestment_periods[sample_period]

    print(f"\n📅 {sample_period.title()} Reinvestment Scenario (${sample_reinvestment:,}):")
    scenario_result = enhanced_investment_analysis_pipeline(
        api_key_openai=OPENAI_API_KEY,
        user_input_keyword=keyword,
        investment_amount=initial_investment,
        reinvestment_amount=sample_reinvestment,
        news_data_path=NEWS_DATA_PATH
    )
    if isinstance(scenario_result, (int, float)):
        roi = ((scenario_result - initial_investment - sample_reinvestment) / (initial_investment + sample_reinvestment)) * 100
        print(f"💎 Projected ROI: {roi:.2f}%")

else:
    print("\n❌ Analysis failed. Please check:")
    print("1. 📁 NY Times news data exists in Google Drive")
    print("   - Run Untitled0.ipynb first to collect the data")
    print("   - Check folder: /MyDrive/Investment_News_Data/")
    print("   - Ensure 'investment_news_data.csv' file exists")
    print("2. 🔑 OpenAI API key is valid")
    print("3. 🌐 Internet connection for stock data validation")
    print("4. 💾 Google Drive is properly mounted (in Colab)")

🚀 Starting Enhanced Investment Analysis with NY Times Data
🔗 Integrating with Google Drive for data access
📱 Google Colab environment detected
💾 Will attempt to access NY Times data from Google Drive

🎯 Analyzing technology sector using NY Times news data...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully!
🔍 Searching for news data in 6 possible locations...
  1. Checking: /content/drive/MyDrive/Investment_News_Data/investment_news_data.csv
  ✅ Found at: /content/drive/MyDrive/Investment_News_Data/investment_news_data.csv
✅ Successfully loaded 198 articles from:
   📁 /content/drive/MyDrive/Investment_News_Data/investment_news_data.csv
📊 Data breakdown:
   - Technology: 84 articles
   - Energy: 67 articles
   - Agriculture: 47 articles
🎯 Filtered to 84 technology articles
✅ Received analysis from OpenAI
✅ Successfully parsed JSON with 17 companies
📊 Extracted 17 company t

/tmp/ipython-input-2152901591.py:262: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-2152901591.py:262: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-2152901591.py:262: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-2152901591.py:262: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-2152901591.py:262: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-2152901591.py:26

✅ Validated 17 out of 17 tickers
📊 Portfolio Return Calculation: 297.85%
📊 Based on 17 successful calculations

📋 ENHANCED INVESTMENT ANALYSIS SUMMARY
🎯 Target Industry: technology
📰 News Articles Analyzed: 84
🏢 Companies Recommended: 17
📊 Portfolio Return Estimate: 297.85%
💰 Investment Amount: $10,000.00
💎 Projected Value: $39,784.93

📊 COMPREHENSIVE COMPANY ANALYSIS RESULTS
🎯 Market Themes: Artificial Intelligence, Cloud Computing, Regulatory Changes
📈 Key Trends: Surge in AI Investments, Increased Demand for Cloud Services, Regulatory Scrutiny on Big Tech

🏢 TOP RECOMMENDED COMPANIES (17 total):
----------------------------------------------------------------------
 1. AAPL - Apple Inc.
    📊 Allocation: 10.0%
    🎯 Confidence: 9.0/10
    💡 Thesis: Apple continues to innovate with its hardware and software integration, maintaining a strong competi...

 2. MSFT - Microsoft Corporation
    📊 Allocation: 12.5%
    🎯 Confidence: 8.8/10
    💡 Thesis: Microsoft's strong cloud computing gr

/tmp/ipython-input-2152901591.py:262: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-2152901591.py:262: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-2152901591.py:262: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-2152901591.py:262: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-2152901591.py:262: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipython-input-2152901591.py:26

⚠️ No data found for SQ
✅ Validated 15 out of 16 tickers
📊 Portfolio Return Calculation: 290.15%
📊 Based on 15 successful calculations

📋 ENHANCED INVESTMENT ANALYSIS SUMMARY
🎯 Target Industry: technology
📰 News Articles Analyzed: 84
🏢 Companies Recommended: 15
📊 Portfolio Return Estimate: 290.15%
💰 Investment Amount: $10,000.00
💎 Projected Value (with reinvestment): $49,015.40
💎 Projected ROI: 145.08%


In [ ]:
# Google Drive Data Verification (Run this first to check your data)
import os
import pandas as pd

def verify_nytimes_data():
    """
    Verify that the NY Times news data is available in Google Drive
    Run this before the main analysis to ensure everything is set up correctly
    """

    print("🔍 VERIFYING NY TIMES NEWS DATA AVAILABILITY")
    print("="*60)

    # Check if we're in Colab and mount Drive
    try:
        from google.colab import drive
        print("📱 Google Colab environment detected")

        # Mount Google Drive
        try:
            drive.mount('/content/drive')
            print("✅ Google Drive mounted successfully!")
        except Exception as e:
            print(f"⚠️ Drive mount status: {e}")

        # Define possible data locations
        drive_paths = [
            "/content/drive/MyDrive/Investment_News_Data/investment_news_data.csv",
            "/content/drive/My Drive/Investment_News_Data/investment_news_data.csv",
            "/content/drive/MyDrive/investment_news_data.csv",
            "/content/drive/My Drive/investment_news_data.csv"
        ]

        print(f"\n📁 Checking Google Drive for data files...")

        data_found = False
        for i, path in enumerate(drive_paths, 1):
            print(f"\n{i}. Checking: {path}")

            if os.path.exists(path):
                print(f"   ✅ FILE FOUND!")
                try:
                    # Try to load and analyze the file
                    df = pd.read_csv(path)
                    print(f"   📊 Contains {len(df)} articles")

                    if len(df) > 0:
                        # Show industry breakdown
                        industries = df['industry'].value_counts()
                        print(f"   🏭 Industries:")
                        for industry, count in industries.items():
                            print(f"      - {industry.title()}: {count} articles")

                        # Show date range
                        if 'pub_date' in df.columns:
                            print(f"   📅 Date range: {df['pub_date'].min()} to {df['pub_date'].max()}")

                        print(f"   💾 This file will be used for analysis!")
                        data_found = True
                        break

                except Exception as e:
                    print(f"   ❌ Error reading file: {e}")
            else:
                print(f"   ❌ Not found")

        if not data_found:
            print(f"\n❌ NO DATA FILES FOUND!")
            print(f"📋 TO FIX THIS:")
            print(f"1. 🔄 Run the Untitled0.ipynb notebook first")
            print(f"2. 📊 Make sure it collects NY Times news data")
            print(f"3. 💾 Ensure data is saved to Google Drive")
            print(f"4. 📁 Check your Drive folder: /MyDrive/Investment_News_Data/")
            print(f"5. 🔍 Look for file: investment_news_data.csv")
            return False
        else:
            print(f"\n✅ DATA VERIFICATION SUCCESSFUL!")
            print(f"🎯 Ready to run investment analysis!")
            return True

    except ImportError:
        print("💻 Local environment detected")
        print("📁 Checking local directories for CSV files...")

        local_paths = [
            "investment_news_data.csv",
            "./investment_news_data.csv",
            "../investment_news_data.csv"
        ]

        for i, path in enumerate(local_paths, 1):
            print(f"{i}. Checking: {path}")
            if os.path.exists(path):
                try:
                    df = pd.read_csv(path)
                    print(f"   ✅ Found! Contains {len(df)} articles")
                    return True
                except Exception as e:
                    print(f"   ❌ Error reading: {e}")
            else:
                print(f"   ❌ Not found")

        print(f"\n❌ No data files found in local directories")
        print(f"📋 Please run Untitled0.ipynb first to collect the data")
        return False

# Run verification
print("🚀 Running data verification before analysis...")
if verify_nytimes_data():
    print("\n🎉 All systems ready! You can now run the investment analysis.")
else:
    print("\n⚠️ Please resolve the data issues before proceeding.")

🚀 Running data verification before analysis...
🔍 VERIFYING NY TIMES NEWS DATA AVAILABILITY
📱 Google Colab environment detected
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully!

📁 Checking Google Drive for data files...

1. Checking: /content/drive/MyDrive/Investment_News_Data/investment_news_data.csv
   ✅ FILE FOUND!
   📊 Contains 198 articles
   🏭 Industries:
      - Technology: 84 articles
      - Energy: 67 articles
      - Agriculture: 47 articles
   📅 Date range: 2022-10-28T09:00:04Z to 2025-09-07T01:21:19Z
   💾 This file will be used for analysis!

✅ DATA VERIFICATION SUCCESSFUL!
🎯 Ready to run investment analysis!

🎉 All systems ready! You can now run the investment analysis.


In [ ]:
# Integration with Financial Data Analysis (FINANCE_REPORT.ipynb style)
import time
import re
from google.colab import userdata  # For Colab secrets

def fetch_financial_data_for_recommended_companies(analysis_results, alpha_vantage_key=None):
    """
    Fetch financial data for the companies recommended by the news analysis
    This integrates the methodology from FINANCE_REPORT.ipynb
    """

    if not analysis_results or not isinstance(analysis_results, dict):
        print("❌ No valid analysis results provided")
        return None

    # Get Alpha Vantage API key
    if alpha_vantage_key is None:
        try:
            alpha_vantage_key = userdata.get("ALPHA_VANTAGE_API_KEY")
        except:
            print("⚠️ Alpha Vantage API key not found. Please add it to Colab secrets or provide directly.")
            return None

    # Extract tickers from the analysis
    recommended_companies = analysis_results.get('analysis_data', {}).get('recommended_companies', [])
    tickers = [company.get('ticker') for company in recommended_companies if company.get('ticker')]

    if not tickers:
        print("❌ No valid tickers found in analysis results")
        return None

    print(f"📊 Fetching financial data for {len(tickers)} companies...")
    print(f"🏢 Companies: {', '.join(tickers[:10])}{'...' if len(tickers) > 10 else ''}")

    # Alpha Vantage API functions
    ALPHA_VANTAGE_BASE_URL = "https://www.alphavantage.co/query"

    def get_alpha_vantage_report(ticker, function, api_key):
        """Fetches financial data from Alpha Vantage."""
        url = f"{ALPHA_VANTAGE_BASE_URL}?function={function}&symbol={ticker}&apikey={api_key}"
        try:
            response = requests.get(url)
            response.raise_for_status()
            data = response.json()
            return data
        except requests.exceptions.RequestException as e:
            print(f"⚠️ Alpha Vantage API error ({function}) for {ticker}: {e}")
            return {}
        except json.JSONDecodeError:
            print(f"⚠️ Could not decode JSON from Alpha Vantage response for {ticker} ({function}).")
            return {}

    # Fetch financial data for all recommended companies
    financial_data = []

    for i, ticker in enumerate(tickers, 1):
        print(f"📈 [{i}/{len(tickers)}] Fetching data for {ticker}...")

        # Fetch the three main financial statements
        income_statement = get_alpha_vantage_report(ticker, "INCOME_STATEMENT", alpha_vantage_key)
        balance_sheet = get_alpha_vantage_report(ticker, "BALANCE_SHEET", alpha_vantage_key)
        cash_flow = get_alpha_vantage_report(ticker, "CASH_FLOW", alpha_vantage_key)

        company_data = {
            "ticker": ticker,
            "income": income_statement.get("quarterlyReports", []),
            "balance_sheet": balance_sheet.get("quarterlyReports", []),
            "cash_flow": cash_flow.get("quarterlyReports", [])
        }
        financial_data.append(company_data)

        # Rate limiting - Alpha Vantage free tier allows 5 calls per minute
        if i < len(tickers):  # Don't sleep after the last request
            print("⏳ Waiting 15 seconds (rate limiting)...")
            time.sleep(15)

    print("✅ Financial data collection complete!")
    return financial_data

def generate_financial_based_strategy(analysis_results, financial_data, openai_key):
    """
    Generate investment strategy based on both news analysis and financial data
    This combines insights from both sources for better decision making
    """

    if not financial_data:
        print("❌ No financial data available for strategy generation")
        return None

    client = OpenAI(api_key=openai_key)

    # Prepare financial data string for OpenAI analysis
    financial_data_string = ""
    for company_data in financial_data:
        financial_data_string += f"Ticker: {company_data['ticker']}\n"

        # Income statement data
        financial_data_string += "Income Statement (Recent Quarters):\n"
        for report in company_data.get('income', [])[:4]:  # Last 4 quarters
            fiscal_date = report.get('fiscalDateEnding', 'N/A')
            revenue = report.get('totalRevenue', 'N/A')
            net_income = report.get('netIncome', 'N/A')
            financial_data_string += f"  {fiscal_date}: Revenue={revenue}, Net Income={net_income}\n"

        # Balance sheet data
        financial_data_string += "Balance Sheet (Recent Quarters):\n"
        for report in company_data.get('balance_sheet', [])[:4]:
            fiscal_date = report.get('fiscalDateEnding', 'N/A')
            assets = report.get('totalAssets', 'N/A')
            liabilities = report.get('totalLiabilities', 'N/A')
            financial_data_string += f"  {fiscal_date}: Assets={assets}, Liabilities={liabilities}\n"

        # Cash flow data
        financial_data_string += "Cash Flow (Recent Quarters):\n"
        for report in company_data.get('cash_flow', [])[:4]:
            fiscal_date = report.get('fiscalDateEnding', 'N/A')
            operating_cf = report.get('operatingCashFlow', 'N/A')
            financial_data_string += f"  {fiscal_date}: Operating CF={operating_cf}\n"

        financial_data_string += "---\n"

    # Get the original news-based analysis
    news_analysis = analysis_results.get('analysis_data', {})
    recommended_companies = news_analysis.get('recommended_companies', [])

    # Create enhanced prompt that combines both analyses
    enhanced_prompt = f"""
    You are an expert financial analyst creating the final investment strategy by combining:
    1. News-based market analysis and company recommendations
    2. Detailed financial statement analysis

    NEWS-BASED ANALYSIS SUMMARY:
    - Market Themes: {news_analysis.get('analysis_summary', {}).get('market_themes', [])}
    - Key Trends: {news_analysis.get('analysis_summary', {}).get('key_trends', [])}
    - Companies Identified: {len(recommended_companies)}

    DETAILED FINANCIAL DATA:
    {financial_data_string}

    TASK: Create a refined investment strategy that:
    1. Validates the news-based recommendations using financial fundamentals
    2. Adjusts allocations based on financial health and performance
    3. Identifies the strongest companies from both perspectives
    4. Provides final portfolio allocation percentages (must sum to 100%)

    Return a JSON object with this structure:
    {{
        "final_strategy": {{
            "validation_summary": "How financial data supports or contradicts news analysis",
            "key_financial_insights": ["insight1", "insight2", "insight3"],
            "implementation_date": "2024-01-01"
        }},
        "final_recommendations": [
            {{
                "ticker": "TICKER",
                "company_name": "Company Name",
                "final_allocation": 15.5,
                "news_score": 8.5,
                "financial_score": 9.0,
                "combined_reasoning": "Why this company makes the final cut based on both analyses",
                "key_financial_metrics": ["Strong revenue growth", "Improving margins"],
                "risk_assessment": "Updated risk analysis based on financials"
            }}
        ],
        "portfolio_summary": {{
            "total_companies": 10,
            "diversification_score": 8.5,
            "expected_annual_return": "10-15%",
            "risk_level": "Moderate",
            "confidence_level": "High"
        }}
    }}
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a senior financial analyst creating final investment recommendations by combining news analysis with financial statement analysis. Always return valid JSON."},
                {"role": "user", "content": enhanced_prompt}
            ],
            temperature=0.2
        )

        response_content = response.choices[0].message.content

        # Clean and parse JSON
        if "```json" in response_content:
            response_content = response_content.split("```json")[1].split("```")[0]
        elif "```" in response_content:
            response_content = response_content.split("```")[1]

        final_strategy = json.loads(response_content.strip())

        print("✅ Final investment strategy generated successfully!")
        return final_strategy

    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing failed: {e}")
        return None
    except Exception as e:
        print(f"❌ Strategy generation failed: {e}")
        return None

# Example usage (uncomment to run):
# if 'initial_results' in locals() and initial_results:
#     print("\n🔄 FETCHING FINANCIAL DATA FOR RECOMMENDED COMPANIES...")
#     financial_data = fetch_financial_data_for_recommended_companies(initial_results)
#
#     if financial_data:
#         print("\n🎯 GENERATING FINAL STRATEGY BASED ON FINANCIAL ANALYSIS...")
#         final_strategy = generate_financial_based_strategy(initial_results, financial_data, OPENAI_API_KEY)
#
#         if final_strategy:
#             print("\n📊 FINAL INVESTMENT STRATEGY:")
#             print(json.dumps(final_strategy, indent=2))
#
#             # Save final strategy
#             with open(f"{keyword}_final_investment_strategy.json", 'w') as f:
#                 json.dump(final_strategy, f, indent=2)
#             print(f"💾 Final strategy saved to {keyword}_final_investment_strategy.json")

print("✅ Financial data integration functions ready!")
print("💡 Uncomment the example usage section to run the complete pipeline!")

✅ Financial data integration functions ready!
💡 Uncomment the example usage section to run the complete pipeline!


In [ ]:
# Batch Analysis - Run Multiple Times and Save to CSV
import csv
import time
from datetime import datetime
import random

def run_batch_analysis(num_runs=10, keywords=None, investment_amounts=None,
                      save_detailed_results=True, output_dir="batch_results"):
    """
    Run the investment analysis multiple times and save results to CSV

    Parameters:
    - num_runs: Number of times to run the analysis
    - keywords: List of keywords to test (default: ['technology', 'energy', 'agriculture'])
    - investment_amounts: List of investment amounts to test (default: [10000])
    - save_detailed_results: Whether to save detailed JSON results for each run
    - output_dir: Directory to save results
    """

    if keywords is None:
        keywords = ['technology', 'energy', 'agriculture']

    if investment_amounts is None:
        investment_amounts = [10000]

    # Create output directory
    import os
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Prepare CSV file
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_filename = f"{output_dir}/batch_analysis_results_{timestamp}.csv"

    # CSV headers
    csv_headers = [
        'run_id', 'timestamp', 'keyword', 'investment_amount', 'reinvestment_amount',
        'analysis_success', 'companies_recommended', 'companies_validated',
        'portfolio_return_pct', 'projected_value', 'roi_pct',
        'market_themes', 'key_trends', 'top_companies', 'top_allocations',
        'sector_allocation', 'risk_level', 'expected_annual_return',
        'news_articles_analyzed', 'execution_time_seconds'
    ]

    print(f"🚀 Starting batch analysis with {num_runs} runs")
    print(f"📊 Testing keywords: {keywords}")
    print(f"💰 Testing investment amounts: {investment_amounts}")
    print(f"📁 Results will be saved to: {csv_filename}")
    print("="*70)

    # Open CSV file for writing
    with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=csv_headers)
        writer.writeheader()

        run_count = 0
        successful_runs = 0

        for run_id in range(1, num_runs + 1):
            # Randomly select keyword and investment amount for variety
            keyword = random.choice(keywords)
            investment_amount = random.choice(investment_amounts)

            # Random reinvestment (0, quarterly, or yearly)
            reinvestment_options = [0, 2500, 10000]
            reinvestment_amount = random.choice(reinvestment_options)

            print(f"\n📈 Run {run_id}/{num_runs}: {keyword} with ${investment_amount:,} investment")
            run_start_time = time.time()

            try:
                # Run the analysis
                result = enhanced_investment_analysis_pipeline(
                    api_key_openai=OPENAI_API_KEY,
                    user_input_keyword=keyword,
                    investment_amount=investment_amount,
                    reinvestment_amount=reinvestment_amount,
                    news_data_path=NEWS_DATA_PATH
                )

                run_time = time.time() - run_start_time
                run_count += 1

                if result and isinstance(result, dict):
                    successful_runs += 1
                    analysis_data = result.get('analysis_data', {})

                    # Extract key metrics
                    summary = analysis_data.get('analysis_summary', {})
                    companies = analysis_data.get('recommended_companies', [])
                    portfolio_strategy = analysis_data.get('portfolio_strategy', {})

                    # Calculate ROI
                    portfolio_return = result.get('portfolio_return', 0)
                    projected_value = investment_amount + (investment_amount * portfolio_return / 100)
                    if reinvestment_amount > 0:
                        projected_value += reinvestment_amount

                    total_invested = investment_amount + reinvestment_amount
                    roi_pct = ((projected_value - total_invested) / total_invested) * 100 if total_invested > 0 else 0

                    # Prepare CSV row
                    csv_row = {
                        'run_id': run_id,
                        'timestamp': datetime.now().isoformat(),
                        'keyword': keyword,
                        'investment_amount': investment_amount,
                        'reinvestment_amount': reinvestment_amount,
                        'analysis_success': True,
                        'companies_recommended': len(companies),
                        'companies_validated': result.get('validated_tickers', []),
                        'portfolio_return_pct': round(portfolio_return, 2),
                        'projected_value': round(projected_value, 2),
                        'roi_pct': round(roi_pct, 2),
                        'market_themes': '; '.join(summary.get('market_themes', [])),
                        'key_trends': '; '.join(summary.get('key_trends', [])),
                        'top_companies': '; '.join([f"{c.get('ticker', '')}({c.get('recommended_allocation', 0)}%)"
                                                  for c in companies[:5]]),
                        'top_allocations': '; '.join([f"{c.get('recommended_allocation', 0)}%"
                                                    for c in companies[:5]]),
                        'sector_allocation': str(portfolio_strategy.get('sector_allocation', {})),
                        'risk_level': portfolio_strategy.get('risk_level', ''),
                        'expected_annual_return': portfolio_strategy.get('expected_annual_return', ''),
                        'news_articles_analyzed': result.get('news_articles_analyzed', 0),
                        'execution_time_seconds': round(run_time, 2)
                    }

                    # Save detailed results if requested
                    if save_detailed_results:
                        detailed_filename = f"{output_dir}/detailed_run_{run_id}_{keyword}_{timestamp}.json"
                        try:
                            import json
                            with open(detailed_filename, 'w', encoding='utf-8') as f:
                                json.dump(result, f, indent=2, default=str)
                        except Exception as e:
                            print(f"⚠️ Could not save detailed results: {e}")

                    print(f"✅ Success: {portfolio_return:.2f}% return, ROI: {roi_pct:.2f}%")

                elif isinstance(result, (int, float)):
                    # Handle case where result is just the final value
                    successful_runs += 1
                    projected_value = result
                    total_invested = investment_amount + reinvestment_amount
                    roi_pct = ((projected_value - total_invested) / total_invested) * 100

                    csv_row = {
                        'run_id': run_id,
                        'timestamp': datetime.now().isoformat(),
                        'keyword': keyword,
                        'investment_amount': investment_amount,
                        'reinvestment_amount': reinvestment_amount,
                        'analysis_success': True,
                        'companies_recommended': 'N/A',
                        'companies_validated': 'N/A',
                        'portfolio_return_pct': 'N/A',
                        'projected_value': round(projected_value, 2),
                        'roi_pct': round(roi_pct, 2),
                        'market_themes': 'N/A',
                        'key_trends': 'N/A',
                        'top_companies': 'N/A',
                        'top_allocations': 'N/A',
                        'sector_allocation': 'N/A',
                        'risk_level': 'N/A',
                        'expected_annual_return': 'N/A',
                        'news_articles_analyzed': 'N/A',
                        'execution_time_seconds': round(run_time, 2)
                    }

                    print(f"✅ Success: Final value ${projected_value:,.2f}, ROI: {roi_pct:.2f}%")

                else:
                    # Failed analysis
                    csv_row = {
                        'run_id': run_id,
                        'timestamp': datetime.now().isoformat(),
                        'keyword': keyword,
                        'investment_amount': investment_amount,
                        'reinvestment_amount': reinvestment_amount,
                        'analysis_success': False,
                        'companies_recommended': 0,
                        'companies_validated': 0,
                        'portfolio_return_pct': 0,
                        'projected_value': investment_amount,
                        'roi_pct': 0,
                        'market_themes': 'FAILED',
                        'key_trends': 'FAILED',
                        'top_companies': 'FAILED',
                        'top_allocations': 'FAILED',
                        'sector_allocation': 'FAILED',
                        'risk_level': 'FAILED',
                        'expected_annual_return': 'FAILED',
                        'news_articles_analyzed': 0,
                        'execution_time_seconds': round(run_time, 2)
                    }

                    print(f"❌ Failed analysis")

                # Write row to CSV
                writer.writerow(csv_row)
                csvfile.flush()  # Ensure data is written immediately

            except Exception as e:
                print(f"❌ Error in run {run_id}: {e}")

                # Write error row
                error_row = {
                    'run_id': run_id,
                    'timestamp': datetime.now().isoformat(),
                    'keyword': keyword,
                    'investment_amount': investment_amount,
                    'reinvestment_amount': reinvestment_amount,
                    'analysis_success': False,
                    'companies_recommended': 0,
                    'companies_validated': 0,
                    'portfolio_return_pct': 0,
                    'projected_value': investment_amount,
                    'roi_pct': 0,
                    'market_themes': f'ERROR: {str(e)[:100]}',
                    'key_trends': 'ERROR',
                    'top_companies': 'ERROR',
                    'top_allocations': 'ERROR',
                    'sector_allocation': 'ERROR',
                    'risk_level': 'ERROR',
                    'expected_annual_return': 'ERROR',
                    'news_articles_analyzed': 0,
                    'execution_time_seconds': round(time.time() - run_start_time, 2)
                }
                writer.writerow(error_row)
                csvfile.flush()

            # Add delay between runs to respect API limits
            if run_id < num_runs:
                print("⏳ Waiting 30 seconds before next run...")
                time.sleep(30)

    # Summary statistics
    print("\n" + "="*70)
    print("📊 BATCH ANALYSIS COMPLETE")
    print("="*70)
    print(f"✅ Total runs completed: {run_count}/{num_runs}")
    print(f"✅ Successful analyses: {successful_runs}/{run_count}")
    print(f"📁 Results saved to: {csv_filename}")

    if save_detailed_results:
        print(f"📁 Detailed JSON results saved to: {output_dir}/")

    # Try to load and show summary statistics
    try:
        import pandas as pd
        df = pd.read_csv(csv_filename)

        if len(df) > 0:
            successful_df = df[df['analysis_success'] == True]

            if len(successful_df) > 0:
                print(f"\n📈 PERFORMANCE STATISTICS:")
                print(f"Average ROI: {successful_df['roi_pct'].mean():.2f}%")
                print(f"Best ROI: {successful_df['roi_pct'].max():.2f}%")
                print(f"Worst ROI: {successful_df['roi_pct'].min():.2f}%")
                print(f"Average Portfolio Return: {successful_df['portfolio_return_pct'].mean():.2f}%")
                print(f"Average Companies per Analysis: {successful_df['companies_recommended'].mean():.1f}")

                print(f"\n🏭 KEYWORD PERFORMANCE:")
                for keyword in keywords:
                    keyword_df = successful_df[successful_df['keyword'] == keyword]
                    if len(keyword_df) > 0:
                        avg_roi = keyword_df['roi_pct'].mean()
                        print(f"{keyword.title()}: {avg_roi:.2f}% average ROI ({len(keyword_df)} runs)")

    except Exception as e:
        print(f"⚠️ Could not generate statistics: {e}")

    return csv_filename

# Quick batch analysis function
def quick_batch_analysis(num_runs=5):
    """Run a quick batch analysis with default settings"""
    return run_batch_analysis(
        num_runs=num_runs,
        keywords=['technology', 'energy', 'agriculture'],
        investment_amounts=[10000],
        save_detailed_results=False
    )

print("✅ Batch analysis functions ready!")
print("💡 Usage examples:")
print("   quick_batch_analysis(5)  # Run 5 quick analyses")
print("   run_batch_analysis(10, ['technology'], [10000, 25000])  # Custom parameters")

✅ Batch analysis functions ready!
💡 Usage examples:
   quick_batch_analysis(5)  # Run 5 quick analyses
   run_batch_analysis(10, ['technology'], [10000, 25000])  # Custom parameters


In [ ]:
# Example: Run Batch Analysis (Uncomment to execute)

# QUICK TEST - Run 3 analyses with default settings
print("🚀 Running quick batch test with 3 iterations...")
csv_file = quick_batch_analysis(3)
print(f"✅ Results saved to: {csv_file}")

# CUSTOM BATCH ANALYSIS - Uncomment and modify as needed
# print("🚀 Running custom batch analysis...")
# csv_file = run_batch_analysis(
#     num_runs=10,                                    # Number of runs
#     keywords=['technology', 'energy', 'agriculture'], # Industries to test
#     investment_amounts=[10000, 25000, 50000],       # Investment amounts to test
#     save_detailed_results=True,                     # Save JSON files for each run
#     output_dir="my_batch_results"                   # Output directory
# )

# LOAD AND ANALYZE RESULTS
# Uncomment to load previous results and perform analysis
import pandas as pd
import matplotlib.pyplot as plt

# Load the CSV results
try:
    df = pd.read_csv(csv_file)

    print("\n📊 BATCH RESULTS ANALYSIS")
    print("="*50)

    # Success rate
    success_rate = (df['analysis_success'].sum() / len(df)) * 100
    print(f"Success Rate: {success_rate:.1f}%")

    # Performance by keyword
    successful_df = df[df['analysis_success'] == True]
    if len(successful_df) > 0:
        print(f"\n🏭 Performance by Industry:")
        keyword_stats = successful_df.groupby('keyword').agg({
            'roi_pct': ['mean', 'std', 'min', 'max'],
            'portfolio_return_pct': ['mean', 'std'],
            'companies_recommended': 'mean'
        }).round(2)
        print(keyword_stats)

        # Best performing runs
        print(f"\n🎯 Top 5 Best ROI Results:")
        best_runs = successful_df.nlargest(5, 'roi_pct')[
            ['keyword', 'investment_amount', 'roi_pct', 'portfolio_return_pct', 'projected_value']
        ]
        print(best_runs.to_string(index=False))

        # Save summary statistics
        summary_file = csv_file.replace('.csv', '_summary.txt')
        with open(summary_file, 'w') as f:
            f.write(f"Batch Analysis Summary\n")
            f.write(f"="*50 + "\n")
            f.write(f"Total Runs: {len(df)}\n")
            f.write(f"Successful Runs: {len(successful_df)}\n")
            f.write(f"Success Rate: {success_rate:.1f}%\n")
            f.write(f"Average ROI: {successful_df['roi_pct'].mean():.2f}%\n")
            f.write(f"Best ROI: {successful_df['roi_pct'].max():.2f}%\n")
            f.write(f"Average Execution Time: {df['execution_time_seconds'].mean():.1f} seconds\n")

        print(f"\n💾 Summary saved to: {summary_file}")

except Exception as e:
    print(f"❌ Could not analyze results: {e}")

print("✅ Batch analysis examples ready!")
print("\n💡 To run batch analysis:")
print("1. Uncomment the quick test section above")
print("2. Or uncomment the custom batch analysis section")
print("3. Run this cell to start the batch process")
print("\n📋 The batch analysis will:")
print("• Run the investment analysis multiple times")
print("• Test different keywords and investment amounts")
print("• Save all results to a timestamped CSV file")
print("• Generate summary statistics")
print("• Optionally save detailed JSON results for each run")
print("\n⚠️ Note: Each run takes ~30-60 seconds due to API calls and rate limiting")

🚀 Running quick batch test with 3 iterations...
🚀 Starting batch analysis with 3 runs
📊 Testing keywords: ['technology', 'energy', 'agriculture']
💰 Testing investment amounts: [10000]
📁 Results will be saved to: batch_results/batch_analysis_results_20251005_072616.csv

📈 Run 1/3: agriculture with $10,000 investment
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted successfully!
🔍 Searching for news data in 6 possible locations...
  1. Checking: /content/drive/MyDrive/Investment_News_Data/investment_news_data.csv
  ✅ Found at: /content/drive/MyDrive/Investment_News_Data/investment_news_data.csv
✅ Successfully loaded 198 articles from:
   📁 /content/drive/MyDrive/Investment_News_Data/investment_news_data.csv
📊 Data breakdown:
   - Technology: 84 articles
   - Energy: 67 articles
   - Agriculture: 47 articles
🎯 Filtered to 47 agriculture articles


KeyboardInterrupt: 